# La³⁺ metadynamics — 100 ps with MACE-polar-1-S

Replicates the instructions gicen by advisor: well-tempered metadynamics on La³⁺ + 3 OH⁻ and
La³⁺ + 3 F⁻ in 128-water droplets, 100 ps each, MACE-polar small model.

Uses the pre-equilibrated droplets from `data/f-block-electrolytes/`. Upload
`La_OH_droplet.xyz` and `La_F_droplet.xyz` to Colab when prompted.

Runs OH⁻ first, then F⁻ sequentially. Checkpoints to Drive every 5 ps.
If Colab disconnects, rerun all cells through Setup, then skip to the Resume cell.

It is a single purpose notebook, no script equivalent. 

In [ ]:
!pip install -q mace-torch ase
!pip install -q git+https://github.com/WillBaldwin0/graph_electrostatics.git@v0.4.0

## Setup

Run every cell in this section after a reconnect.

In [ ]:
import time
from pathlib import Path
import numpy as np
import torch
from ase import Atoms
from ase.io import read, write
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.calculators.calculator import Calculator, all_changes
import ase.units as units
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/gdrive')

DRIVE_DIR = Path('/content/gdrive/MyDrive/La_Metadynamics_100ps')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'{DEVICE} {torch.cuda.get_device_name() if DEVICE == "cuda" else ""}')

KJ_TO_EV = 1.0 / 96.485
EV_TO_KJ = 96.485

SYSTEMS = {
    'oh': {'label': 'La3+ + 3 OH-', 'r0': 3.5, 'element': 'O', 'n_anions': 3},
    'f':  {'label': 'La3+ + 3 F-',  'r0': 3.2, 'element': 'F', 'n_anions': 3},
}

In [ ]:
def get_calculator():
    from mace.calculators import mace_polar
    calc = mace_polar(model='polar-1-s', device=DEVICE, default_dtype='float64')
    print(f'Loaded MACE-polar (polar-1-s) on {DEVICE}')
    return calc

def smooth_cn_and_grad(positions, la_idx, anion_indices, r0, n=6, m=12):
    la_pos = positions[la_idx]
    cn = 0.0
    grad = np.zeros((len(positions), 3))
    for i in anion_indices:
        r_vec = positions[i] - la_pos
        r = np.linalg.norm(r_vec)
        if r < 1e-10: continue
        r_hat = r_vec / r
        u = r / r0
        u_n, u_m = u**n, u**m
        denom = max(1.0 - u_m, 1e-12)
        cn += (1.0 - u_n) / denom
        dcn_dr = (1.0/r0) * (-n*u**(n-1)*(1.0-u_m) + m*u**(m-1)*(1.0-u_n)) / denom**2
        grad[i] += dcn_dr * r_hat
        grad[la_idx] -= dcn_dr * r_hat
    return cn, grad

class WellTemperedMetadynamics:
    def __init__(self):
        self.sigma = 0.15
        self.w0 = 2.0 * KJ_TO_EV
        self.pace = 500
        self.gamma = 15
        self.kBT = 8.617e-5 * 300
        self.hills = []
        self.colvar = []
        self.step = 0

    def bias_potential(self, cn):
        v = 0.0
        for cn_k, w_k in self.hills:
            v += w_k * np.exp(-(cn - cn_k)**2 / (2*self.sigma**2))
        return v

    def bias_gradient(self, cn):
        dv = 0.0
        for cn_k, w_k in self.hills:
            g = np.exp(-(cn - cn_k)**2 / (2*self.sigma**2))
            dv += w_k * g * (-(cn - cn_k) / self.sigma**2)
        return dv

    def deposit_hill(self, cn):
        v = self.bias_potential(cn)
        w = self.w0 * np.exp(-v / (self.kBT * (self.gamma - 1)))
        self.hills.append((cn, w))
        return w

    def record_colvar(self, step, cn, energy, temp):
        self.colvar.append((step, cn, self.bias_potential(cn), energy, temp))

    def save(self, path):
        path = Path(path)
        hills_arr = np.array(self.hills) if self.hills else np.empty((0,2))
        colvar_arr = np.array(self.colvar) if self.colvar else np.empty((0,5))
        np.savez(path / 'metad_state.npz',
                 sigma=self.sigma, w0_eV=self.w0, pace=self.pace,
                 gamma=self.gamma, kBT_eV=self.kBT, step=self.step,
                 hills=hills_arr, colvar=colvar_arr)
        if self.hills:
            steps = np.arange(1, len(hills_arr)+1) * self.pace
            np.savetxt(path / 'HILLS.txt',
                       np.column_stack([steps, hills_arr]),
                       header='step cn weight_eV', fmt='%.0f %.6f %.8f')
        if self.colvar:
            np.savetxt(path / 'COLVAR.txt', colvar_arr,
                       header='step cn bias_eV energy_eV temp_K',
                       fmt='%.0f %.6f %.6f %.4f %.1f')

    def load(self, path):
        data = np.load(Path(path) / 'metad_state.npz')
        self.sigma = float(data['sigma'])
        self.w0 = float(data['w0_eV'])
        self.pace = int(data['pace'])
        self.gamma = float(data['gamma'])
        self.kBT = float(data['kBT_eV'])
        self.step = int(data['step'])
        h = data['hills']
        self.hills = [(float(r[0]), float(r[1])) for r in h] if len(h) else []
        c = data['colvar']
        self.colvar = [tuple(r) for r in c] if len(c) else []

class BiasedCalculator(Calculator):
    implemented_properties = ['energy', 'forces']
    def __init__(self, base_calc, metad, la_idx, anion_indices, r0):
        super().__init__()
        self.base_calc = base_calc
        self.metad = metad
        self.la_idx = la_idx
        self.anion_indices = anion_indices
        self.r0 = r0
        self.last_cn = 0.0
    def calculate(self, atoms=None, properties=None, system_changes=all_changes):
        if properties is None: properties = ['energy', 'forces']
        super().calculate(atoms, properties, system_changes)
        self.base_calc.calculate(atoms, properties, system_changes)
        base_e = self.base_calc.results['energy']
        base_f = self.base_calc.results['forces'].copy()
        cn, dcn = smooth_cn_and_grad(atoms.positions, self.la_idx, self.anion_indices, self.r0)
        v = self.metad.bias_potential(cn)
        dv = self.metad.bias_gradient(cn)
        self.results['energy'] = base_e + v
        self.results['forces'] = base_f - dv * dcn
        self.last_cn = cn

In [ ]:
def identify_atoms(atoms, system_type):
    symbols = atoms.get_chemical_symbols()
    cfg = SYSTEMS[system_type]
    la_idx = [i for i, s in enumerate(symbols) if s == 'La'][0]
    la_pos = atoms.positions[la_idx]
    candidates = sorted(
        [(i, np.linalg.norm(atoms.positions[i] - la_pos))
         for i, s in enumerate(symbols) if s == cfg['element']],
        key=lambda x: x[1])
    return la_idx, [idx for idx, _ in candidates[:cfg['n_anions']]]

def save_checkpoint(atoms, metad, name):
    d = DRIVE_DIR / name
    d.mkdir(parents=True, exist_ok=True)
    for old in d.glob('ckpt_*.xyz'): old.unlink()
    write(d / f'ckpt_{metad.step}.xyz', atoms)
    metad.save(d)

def load_checkpoint(name):
    d = DRIVE_DIR / name
    if not d.exists(): return None
    ckpts = sorted(d.glob('ckpt_*.xyz'))
    if not ckpts: return None
    step = int(ckpts[-1].stem.split('_')[1])
    atoms = read(ckpts[-1])
    metad = WellTemperedMetadynamics()
    metad.load(d)
    print(f'Resumed {name} from step {step} ({step/1000:.1f} ps), {len(metad.hills)} hills')
    return atoms, metad, step

def build_droplet(system_type, n_waters=128):
    cfg = SYSTEMS[system_type]
    positions = [[0,0,0]]
    symbols = ['La']
    r = 2.45
    dirs = [[1,0,0], [-0.5,0.866,0], [-0.5,-0.866,0]]
    if system_type == 'oh':
        for d in dirs:
            positions.append([r*x for x in d]); symbols.append('O')
            positions.append([(r+0.96)*x for x in d]); symbols.append('H')
    else:
        for d in dirs:
            positions.append([r*x for x in d]); symbols.append('F')
    rng = np.random.default_rng(42)
    placed = 0
    for _ in range(n_waters * 500):
        if placed >= n_waters: break
        rad = 10.0 * rng.random()**(1/3)
        if rad < 3.5: continue
        th = np.arccos(2*rng.random()-1)
        ph = 2*np.pi*rng.random()
        o = np.array([rad*np.sin(th)*np.cos(ph), rad*np.sin(th)*np.sin(ph), rad*np.cos(th)])
        if any(np.linalg.norm(o - np.array(p)) < 2.2 for p in positions): continue
        ax = rng.standard_normal(3); ax /= np.linalg.norm(ax)
        a = rng.random()*2*np.pi
        ca, sa = np.cos(a), np.sin(a)
        K = np.array([[0,-ax[2],ax[1]],[ax[2],0,-ax[0]],[-ax[1],ax[0],0]])
        R = np.eye(3) + sa*K + (1-ca)*K@K
        ha = 52.25*np.pi/180
        h1 = np.array([0.96*np.sin(ha), 0, 0.96*np.cos(ha)])
        h2 = np.array([-0.96*np.sin(ha), 0, 0.96*np.cos(ha)])
        positions.append(o.tolist()); symbols.append('O')
        positions.append((o + R@h1).tolist()); symbols.append('H')
        positions.append((o + R@h2).tolist()); symbols.append('H')
        placed += 1
    atoms = Atoms(symbols=symbols, positions=positions, pbc=False)
    print(f'Built {cfg["label"]} droplet: {len(atoms)} atoms, {placed} waters')
    return atoms

def reconstruct_fes(hills, sigma, cn_grid, gamma):
    fes = np.zeros_like(cn_grid)
    for cn_k, w_k in hills:
        fes += w_k * np.exp(-(cn_grid - cn_k)**2 / (2*sigma**2))
    fes *= -(gamma / (gamma - 1))
    fes -= fes.min()
    return fes * EV_TO_KJ

In [ ]:
def run_metadynamics(system_type, total_steps=100000, start_step=0, metad=None, atoms=None):
    """Run or resume metadynamics for one system. 100 ps = 100000 steps at 1 fs."""
    cfg = SYSTEMS[system_type]
    if atoms is None:
        atoms = build_droplet(system_type)
    atoms.pbc = False
    la_idx, anion_indices = identify_atoms(atoms, system_type)
    print(f'System: {cfg["label"]}, La={la_idx}, anions={anion_indices}, r0={cfg["r0"]} A')
    print(f'Steps: {start_step} -> {total_steps} ({(total_steps-start_step)/1000:.0f} ps)')

    base_calc = get_calculator()
    if metad is None:
        metad = WellTemperedMetadynamics()

    biased = BiasedCalculator(base_calc, metad, la_idx, anion_indices, cfg['r0'])
    atoms.calc = biased

    if start_step == 0:
        MaxwellBoltzmannDistribution(atoms, temperature_K=300)

    dyn = Langevin(atoms, timestep=1.0*units.fs, temperature_K=300, friction=0.01/units.fs)

    t0 = time.time()
    step = start_step
    while step < total_steps:
        dyn.run(1)
        step += 1
        metad.step = step
        cn = biased.last_cn

        if step % 500 == 0:
            metad.deposit_hill(cn)
            metad.record_colvar(step, cn, atoms.get_potential_energy(),
                                atoms.get_temperature())

        if step % 1000 == 0:
            elapsed = time.time() - t0
            rate = (step - start_step) / elapsed if elapsed > 0 else 0
            eta_h = (total_steps - step) / rate / 3600 if rate > 0 else 0
            print(f'  step {step:>7d}/{total_steps}  CN={cn:.3f}  '
                  f'hills={len(metad.hills)}  {rate:.1f} st/s  ETA {eta_h:.1f}h')

        if step % 5000 == 0:
            save_checkpoint(atoms, metad, system_type)

    save_checkpoint(atoms, metad, system_type)
    elapsed = time.time() - t0
    print(f'Done: {step/1000:.0f} ps in {elapsed/3600:.1f} h, {len(metad.hills)} hills')
    return metad

print('Setup complete')

## Upload starting structures

Upload `La_OH_droplet.xyz` and `La_F_droplet.xyz` from `data/f-block-electrolytes/`.
These are pre-equilibrated droplets given.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select both La_OH_droplet.xyz and La_F_droplet.xyz
print(f'Uploaded: {list(uploaded.keys())}')

## Run OH⁻ (100 ps)

In [ ]:
atoms_oh = read('La_OH_droplet.xyz')
print(f'Loaded La_OH_droplet.xyz: {len(atoms_oh)} atoms')
metad_oh = run_metadynamics('oh', atoms=atoms_oh)

## Run F⁻ (100 ps)

In [ ]:
atoms_f = read('La_F_droplet.xyz')
print(f'Loaded La_F_droplet.xyz: {len(atoms_f)} atoms')
metad_f = run_metadynamics('f', atoms=atoms_f)

## Analysis

In [ ]:
cn_grid = np.linspace(-0.2, 3.5, 600)
colors = {'oh': '#5CC2E1', 'f': '#CB62BB'}
labels = {'oh': 'La³⁺ + 3 OH⁻', 'f': 'La³⁺ + 3 F⁻'}

results = {}
for st in SYSTEMS:
    d = DRIVE_DIR / st
    if not d.exists(): continue
    m = WellTemperedMetadynamics()
    try: m.load(d)
    except: continue
    if m.hills: results[st] = m

print(f'Loaded: {list(results.keys())}')

In [ ]:
# FES comparison
fig, ax = plt.subplots(figsize=(7, 4.5))
for st, m in results.items():
    fes = reconstruct_fes(m.hills, m.sigma, cn_grid, m.gamma)
    n_ps = len(m.colvar) * m.pace / 1000
    ax.plot(cn_grid, fes, color=colors[st], lw=2.2, label=f'{labels[st]} ({n_ps:.0f} ps)')
ax.set_xlabel('Coordination Number (La–anion)')
ax.set_ylabel('F(CN) (kJ/mol)')
ax.set_title('Free Energy Surface: La³⁺–OH⁻ vs La³⁺–F⁻ Coordination')
ax.set_xlim(-0.1, 3.2); ax.set_ylim(bottom=0)
ax.legend(frameon=False)
fig.savefig(str(DRIVE_DIR / 'fes_comparison.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# CN time series
n_sys = len(results)
fig, axes = plt.subplots(n_sys, 1, figsize=(7, 3*n_sys), squeeze=False)
for ax_row, (st, m) in zip(axes, results.items()):
    ax = ax_row[0]
    colvar = np.array(m.colvar)
    ax.plot(colvar[:,0]/1000, colvar[:,1], color=colors[st], lw=0.8, alpha=0.8)
    ax.set_ylabel(f'CN (La–{SYSTEMS[st]["element"]})')
    ax.set_title(f'{labels[st]} ({colvar[-1,0]/1000:.0f} ps)')
    ax.set_ylim(-0.1, 3.2)
    for y in [1,2,3]: ax.axhline(y=y, color='#E6E6E7', ls='--', lw=0.6)
axes[-1][0].set_xlabel('Time (ps)')
fig.savefig(str(DRIVE_DIR / 'cn_timeseries.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Convergence
fig, axes = plt.subplots(1, n_sys, figsize=(5*n_sys, 4), sharey=True, squeeze=False)
for ax, (st, m) in zip(axes[0], results.items()):
    for frac, alpha, ls in [(0.33, 0.4, '--'), (0.66, 0.65, '-.'), (1.0, 1.0, '-')]:
        k = max(1, int(len(m.hills)*frac))
        fes = reconstruct_fes(m.hills[:k], m.sigma, cn_grid, m.gamma)
        ps = int(k * m.pace / 1000)
        ax.plot(cn_grid, fes, color=colors[st], lw=1.6, alpha=alpha, ls=ls, label=f'{ps} ps')
    ax.set_title(f'{labels[st]} convergence')
    ax.set_xlabel('CN')
    ax.legend(frameon=False, fontsize=9)
axes[0][0].set_ylabel('F(CN) (kJ/mol)')
fig.savefig(str(DRIVE_DIR / 'fes_convergence.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Validation dashboard
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

ax = axes[0, 0]
for st, m in results.items():
    hills_arr = np.array(m.hills)
    steps_ps = np.arange(1, len(hills_arr)+1) * m.pace / 1000
    ax.scatter(steps_ps, hills_arr[:,1]*EV_TO_KJ, s=12, color=colors[st], alpha=0.7, label=labels[st])
ax.axhline(y=2.0, color='#929295', ls='--', lw=0.8, label='w₀ = 2.0 kJ/mol')
ax.set_xlabel('Time (ps)'); ax.set_ylabel('Hill height (kJ/mol)')
ax.set_title('(a) Well-tempered hill height decay'); ax.set_ylim(0, 2.2)
ax.legend(frameon=False, fontsize=8)

ax = axes[0, 1]
for st, m in results.items():
    colvar = np.array(m.colvar)
    ax.hist(colvar[:,1], bins=30, range=(0,3.2), alpha=0.55, color=colors[st], label=labels[st], density=True)
ax.set_xlabel('Coordination Number'); ax.set_ylabel('Probability density')
ax.set_title('(b) CN exploration (biased)'); ax.legend(frameon=False, fontsize=8)

ax = axes[1, 0]
for st, m in results.items():
    fes = reconstruct_fes(m.hills, m.sigma, cn_grid, m.gamma)
    n_ps = len(m.colvar) * m.pace / 1000
    ax.plot(cn_grid, fes, color=colors[st], lw=2.2, label=f'{labels[st]} ({n_ps:.0f} ps)')
if 'oh' in results:
    fes_oh = reconstruct_fes(results['oh'].hills, results['oh'].sigma, cn_grid, results['oh'].gamma)
    ax.annotate('La(OH)₃ stable\nCN ≈ 3', xy=(2.7, fes_oh[np.argmin(np.abs(cn_grid-2.7))]),
                xytext=(2.8, max(fes_oh)*0.7), fontsize=7.5, color='#6F6F72',
                arrowprops=dict(arrowstyle='-', color='#929295', lw=0.7))
if 'f' in results:
    fes_f = reconstruct_fes(results['f'].hills, results['f'].sigma, cn_grid, results['f'].gamma)
    ax.annotate('F⁻ dissociates\nin water', xy=(0.9, fes_f[np.argmin(np.abs(cn_grid-0.9))]),
                xytext=(0.2, max(fes_f)*0.6), fontsize=7.5, color='#6F6F72',
                arrowprops=dict(arrowstyle='-', color='#929295', lw=0.7))
ax.set_xlabel('Coordination Number (La–anion)'); ax.set_ylabel('F(CN) (kJ/mol)')
ax.set_title('(c) Free energy surface comparison')
ax.set_xlim(-0.1, 3.3); ax.set_ylim(bottom=0); ax.legend(frameon=False, fontsize=8)

ax = axes[1, 1]
for st, m in results.items():
    colvar = np.array(m.colvar)
    ax.plot(colvar[:,0]/1000, colvar[:,2]*EV_TO_KJ, color=colors[st], lw=1.2, label=labels[st])
ax.set_xlabel('Time (ps)'); ax.set_ylabel('V_bias at current CN (kJ/mol)')
ax.set_title('(d) Bias potential growth'); ax.legend(frameon=False, fontsize=8)

fig.tight_layout()
fig.savefig(str(DRIVE_DIR / 'fes_validation_dashboard.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Summary
print('=' * 60)
print('  FES ANALYSIS SUMMARY')
print('=' * 60)
for st, m in results.items():
    fes = reconstruct_fes(m.hills, m.sigma, cn_grid, m.gamma)
    colvar = np.array(m.colvar)
    print(f'\n{labels[st]}:')
    print(f'  Time: {colvar[-1,0]/1000:.1f} ps, {len(m.hills)} hills')
    print(f'  CN explored: [{colvar[:,1].min():.2f}, {colvar[:,1].max():.2f}]')
    print(f'  FES minimum at CN = {cn_grid[np.argmin(fes)]:.2f}')
    if st == 'oh':
        dF = fes[np.argmin(np.abs(cn_grid-1.5))] - fes[np.argmin(np.abs(cn_grid-2.7))]
    else:
        dF = fes[np.argmin(np.abs(cn_grid-0.5))] - fes[np.argmin(np.abs(cn_grid-2.5))]
    print(f'  dF(bound->free) = {dF:+.1f} kJ/mol')
print(f'\nPlots saved to {DRIVE_DIR}')